In [ ]:
!git clone https://github.com/liturriago/CrowdSegmenter-CT-GCE.git
%cd CrowdSegmenter-CT-GCE
!pip install -e ".[training,tuning,viz]" -q

In [ ]:
import kagglehub

path = kagglehub.dataset_download("lucasiturriago/retinal-fundus-glaucoma-analysis")
print("Path to dataset files:", path)

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

In [ ]:
import shlex

# Pon en None cualquier modelo DL que no quieras visualizar (basta con 1)
CTGCE_WEIGHTS = "/content/drive/MyDrive/CrowdSegmenter-CT-GCE/results/ctgce/riga/ct_gce_final.pth"
TGCE_SSPS_WEIGHTS = "/content/drive/MyDrive/CrowdSegmenter-CT-GCE/results/tgce_ssps/riga/tgce_ssps_final.pth"
CROWDSEG_WEIGHTS = "/content/drive/MyDrive/CrowdSegmenter-CT-GCE/results/crowdseg/riga/crowdseg_final.pth"

INCLUDE_MV = True
INCLUDE_STAPLE = True
METHODS = None
SPLIT = "test"
NUM_SAMPLES = 6
SAMPLE_INDICES = None  # Opcional, ej: [0, 5, 12]
SAVE_NPZ = True
OUTPUT_DIR = "/content/drive/MyDrive/CrowdSegmenter-CT-GCE/results/visualizations/riga"

extra_flags = []
for flag, val in (
    ("--ctgce_weights", CTGCE_WEIGHTS),
    ("--tgce_ssps_weights", TGCE_SSPS_WEIGHTS),
    ("--crowdseg_weights", CROWDSEG_WEIGHTS),
):
    if val:
        extra_flags.extend([flag, shlex.quote(val)])
if not INCLUDE_MV:
    extra_flags.append("--no-mv")
if not INCLUDE_STAPLE:
    extra_flags.append("--no-staple")
if METHODS:
    extra_flags.append("--methods")
    extra_flags.extend(shlex.quote(m) for m in METHODS)
if SAMPLE_INDICES is not None:
    extra_flags.append("--sample_indices")
    extra_flags.extend(str(i) for i in SAMPLE_INDICES)
if SAVE_NPZ:
    extra_flags.append("--save-npz")
extra_cli = " ".join(extra_flags)

!python scripts/visualize_predictions.py --dataset riga --data "{path}" --split {SPLIT} --num_samples {NUM_SAMPLES} --output_dir "{OUTPUT_DIR}" {extra_cli}

In [ ]:
from pathlib import Path
from IPython.display import Image, display

for fig_file in sorted(Path(OUTPUT_DIR).glob("sample_*.png")):
    print(fig_file.name)
    display(Image(filename=str(fig_file)))